# ScienceQA — QLoRA Fine-tuning
`SmolVLM-500M-Instruct` · 4-bit NF4 · LoRA adapters

**Workflow:** Edit CONFIG in Cell 2 → Run All → adapter + results auto-saved to Drive.

**Resume:** set `resume_from` to a checkpoint path to continue from that epoch.

In [1]:
# ── CELL 1 · SETUP ────────────────────────────────────────────────────────────
%%capture
!pip install -q transformers==4.51.3 peft==0.15.1 accelerate==1.6.0 \
             bitsandbytes==0.45.5 pillow tqdm matplotlib

from google.colab import drive
drive.mount('/content/drive')


In [2]:
# ── CELL 2 · CONFIG — only cell you edit between runs ─────────────────────────
# run_05: run_02 base + caption augmentation + forced-letter prompt

CONFIG = {
    "run_id":        "run_09",
    "resume_from":   None,
    "model_id":      "HuggingFaceTB/SmolVLM-500M-Instruct",
    "data_dir":      "/content/drive/MyDrive/Data/pixels-to-predictions",
    "output_dir":    "/content/drive/MyDrive/scienceqa_runs",
    "lora_targets":  ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    "lora_r":        16,
    "lora_alpha":    32,
    "lora_dropout":  0.05,
    "learning_rate": 2e-4,
    "num_epochs":    2,
    "batch_size":    8,
    "grad_accum":    4,
    "img_size":      336,
    "max_length":    2048,
    "weight_decay":  0.01,
    "warmup_ratio":  0.05,
    "max_grad_norm": 1.0,
    "quick_val_n":   150,
    "seed":          42,
}

print(f"Run       : {CONFIG['run_id']}")
print(f"LoRA      : r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, targets={CONFIG['lora_targets']}")
print(f"Train     : lr={CONFIG['learning_rate']}, epochs={CONFIG['num_epochs']}, "
      f"batch={CONFIG['batch_size']}x{CONFIG['grad_accum']}={CONFIG['batch_size']*CONFIG['grad_accum']} eff")
print(f"Additions : caption augmentation + forced-letter prompt")


Run       : run_09
LoRA      : r=16, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
Train     : lr=0.0002, epochs=2, batch=8x4=32 eff
Additions : caption augmentation + forced-letter prompt


In [ ]:
# ── CELL 2.5 · CAPTION PREPROCESSING ─────────────────────────────────────────
# Generates 2-3 sentence captions for all images using SmolVLM itself.
# Runs once, saves to captions.csv on Drive. Skip if already exists.
# Takes ~30-40 min on A100 for all 5165 images.

import json, gc
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

DATA_DIR    = Path(CONFIG['data_dir'])
CAPTION_CSV = DATA_DIR / 'captions.csv'

# Check if captions already exist and are non-empty
if CAPTION_CSV.exists():
    existing = pd.read_csv(CAPTION_CSV)
    n_nonempty = (existing['caption'].fillna('').astype(str).str.strip() != '').sum()
    if n_nonempty > 100:
        print(f"Captions already exist: {n_nonempty}/{len(existing)} non-empty — skipping")
    else:
        print(f"Captions file exists but only {n_nonempty} non-empty — regenerating")
        CAPTION_CSV.unlink()

if not CAPTION_CSV.exists():
    # Verify GPU is available
    assert torch.cuda.is_available(), "GPU not available! Change runtime type to GPU first."
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    cap_processor = AutoProcessor.from_pretrained(CONFIG['model_id'])

    # Load in FP16 (not 4-bit) for generation quality
    cap_model = AutoModelForVision2Seq.from_pretrained(
        CONFIG['model_id'],
        torch_dtype=torch.float16,
        device_map='cuda',        # explicitly put on GPU
        low_cpu_mem_usage=True,
    )
    cap_model.eval()
    print(f"Model loaded on: {next(cap_model.parameters()).device}")

    CAPTION_PROMPT = (
        "Describe this image in 2-3 sentences. "
        "Focus on any diagrams, charts, maps, scientific content, "
        "or visual information relevant to answering a science question."
    )

    rows = []
    errors = 0

    BATCH_SIZE = 32  # A100 40GB

    for split in ['train', 'val', 'test']:
        df = pd.read_csv(DATA_DIR / f'{split}.csv')
        df['choices'] = df['choices'].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x)

        for batch_start in tqdm(range(0, len(df), BATCH_SIZE), desc=f'Captions {split}'):
            batch = df.iloc[batch_start:batch_start + BATCH_SIZE]

            imgs, valid_ids, failed_ids = [], [], []
            for _, row in batch.iterrows():
                try:
                    img_path = DATA_DIR / 'images' / row['image_path']
                    img = Image.open(img_path).convert('RGB')
                    img.thumbnail((336, 336), Image.LANCZOS)
                    imgs.append(img)
                    valid_ids.append(row['id'])
                except Exception:
                    failed_ids.append(row['id'])
                    rows.append({'id': row['id'], 'caption': ''})

            if not imgs:
                continue

            try:
                prompts = [cap_processor.apply_chat_template(
                    [{"role": "user", "content": [
                        {"type": "image"},
                        {"type": "text", "text": CAPTION_PROMPT},
                    ]}], add_generation_prompt=True
                ) for _ in imgs]

                inputs = cap_processor(
                    text=prompts, images=imgs,
                    return_tensors='pt', padding=True,
                    max_length=2048, truncation=True,
                )
                inputs = {k: v.to('cuda') for k, v in inputs.items()}

                with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
                    out = cap_model.generate(
                        **inputs,
                        max_new_tokens=80,
                        do_sample=False,
                        pad_token_id=cap_processor.tokenizer.eos_token_id,
                    )

                prompt_len = inputs['input_ids'].shape[1]
                for i, img_id in enumerate(valid_ids):
                    new_tokens = out[i][prompt_len:]
                    caption = cap_processor.tokenizer.decode(
                        new_tokens, skip_special_tokens=True).strip()
                    rows.append({'id': img_id, 'caption': caption})

            except Exception as e:
                errors += len(imgs)
                for img_id in valid_ids:
                    rows.append({'id': img_id, 'caption': ''})
                if errors <= 3:
                    print(f"Batch error: {type(e).__name__}: {e}")

        cap_df = pd.DataFrame(rows)
        cap_df.to_csv(CAPTION_CSV, index=False)
        n_ok = (cap_df['caption'].str.strip() != '').sum()
        print('\n' + f"✓ Saved {len(cap_df)} captions → {CAPTION_CSV}")
        print(f"Non-empty: {n_ok}/{len(cap_df)}  |  Errors: {errors}")

        # Show sample
        sample = cap_df[cap_df['caption'].str.strip() != ''].head(3)
        for _, r in sample.iterrows():
            print(f"  [{r['id']}] {r['caption'][:100]}")

        del cap_model, cap_processor
        gc.collect()
        torch.cuda.empty_cache()
        print("GPU memory freed. Ready for training.")

Captions already exist: 3026/7266 non-empty — skipping


In [ ]:
# ── CELL 3 · TRAIN ────────────────────────────────────────────────────────────
import json, random, time, math, shutil, gc
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from transformers import (
    AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
)
from peft import (
    LoraConfig, get_peft_model, TaskType,
    prepare_model_for_kbit_training, PeftModel
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Seed & device ─────────────────────────────────────────────────────────────
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
torch.cuda.manual_seed_all(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    COMPUTE_DTYPE = torch.bfloat16
    USE_SCALER    = False
    print("Mixed precision: BF16 (A100)")
else:
    COMPUTE_DTYPE = torch.float16
    USE_SCALER    = True
    print("Mixed precision: FP16 (T4/P100)")

scaler = GradScaler('cuda', enabled=USE_SCALER)
print(f"Device: {DEVICE} · {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = Path(CONFIG['data_dir'])
OUT_DIR     = Path(CONFIG['output_dir']) / CONFIG['run_id']
OUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = OUT_DIR / f"adapter_{CONFIG['run_id']}"
LOG_FILE    = OUT_DIR / "train_log.jsonl"
CAPTION_CSV = DATA_DIR / 'captions.csv'

# ── Safety guard ──────────────────────────────────────────────────────────────
if ADAPTER_DIR.exists() and not CONFIG['resume_from']:
    raise RuntimeError(
        f"\u26a0\ufe0f  {ADAPTER_DIR} already exists!\n"
        f"    \u2192 Change run_id, or set resume_from to continue."
    )

# ── Resume epoch detection ────────────────────────────────────────────────────
if CONFIG['resume_from']:
    completed = []
    for d in OUT_DIR.iterdir():
        if d.is_dir() and d.name.startswith('checkpoint_epoch'):
            try: completed.append(int(d.name.replace('checkpoint_epoch', '')))
            except ValueError: pass
    START_EPOCH = max(completed) if completed else 0
    print(f"Resuming from epoch {START_EPOCH + 1}  (found: {sorted(completed)})")
else:
    START_EPOCH = 0

# ── Data ──────────────────────────────────────────────────────────────────────
def load_df(name):
    df = pd.read_csv(DATA_DIR / f'{name}.csv')
    df['choices'] = df['choices'].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x)
    return df

train_df = load_df('train')
val_df   = load_df('val')
print(f"Train: {len(train_df):,} · Val: {len(val_df):,}")

# ── Load captions ─────────────────────────────────────────────────────────────
if CAPTION_CSV.exists():
    captions_df = pd.read_csv(CAPTION_CSV).set_index('id')
    n_caps = (captions_df['caption'].fillna('') != '').sum()
    print(f"Captions loaded: {n_caps}/{len(captions_df)} non-empty")
else:
    captions_df = None
    print("\u26a0  No captions.csv found — run Cell 2.5 first for best results")

# ── Processor ────────────────────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(CONFIG['model_id'])
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# ── Prompt ────────────────────────────────────────────────────────────────────
MAX_LECTURE_CHARS = 800

def build_prompt(row, include_answer=True):
    parts = []

    # Grade + subject
    grade   = row.get('grade', '')
    subject = row.get('subject', '')
    if grade and str(grade) not in ('', 'nan'):
        parts.append(f"Grade {grade} \u00b7 {subject}")

    # Caption augmentation
    if captions_df is not None:
        try:
            cap = captions_df.loc[row['id'], 'caption']
            if isinstance(cap, pd.Series):
                cap = cap.iloc[0]
            if cap and str(cap) not in ('', 'nan'):
                parts.append(f"Caption: {str(cap).strip()}")
        except KeyError:
            pass

    # Context: hint first, lecture second (truncated)
    hint    = str(row.get('hint',    '')).strip()
    lecture = str(row.get('lecture', '')).strip()
    hint    = '' if hint    in ('', 'nan') else hint
    lecture = '' if lecture in ('', 'nan') else lecture[:MAX_LECTURE_CHARS]
    ctx = [c for c in [hint, lecture] if c]
    if ctx:
        parts.append('Context:\n' + '\n'.join(ctx))

    parts.append(f"Question: {row['question']}")
    parts.append('Choices:\n' + '\n'.join(
        f'  {chr(65+i)}. {c}' for i, c in enumerate(row['choices'])))

    # Forced-letter prompt (LLaVA-1.5 standard)
    parts.append('Answer with the option\u2019s letter from the given choices directly.')
    parts.append('Answer:')
    if include_answer:
        parts.append(chr(65 + int(row['answer'])))

    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": '\n'.join(parts)},
    ]}]
    return processor.apply_chat_template(
        messages, add_generation_prompt=not include_answer)

# ── Image loading ─────────────────────────────────────────────────────────────
def load_image(image_path, size):
    try:
        img = Image.open(DATA_DIR / image_path).convert('RGB')
        img.thumbnail((size, size), Image.LANCZOS)
        padded = Image.new('RGB', (size, size), (128, 128, 128))
        padded.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
        return padded
    except Exception:
        return Image.new('RGB', (size, size), (128, 128, 128))

# ── Dataset ───────────────────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = load_image(row['image_path'], CONFIG['img_size'])
        ans = int(row['answer']) if pd.notna(row.get('answer')) else 0
        ans = min(ans, 4)
        return {'image': img, 'text': build_prompt(row, include_answer=True), 'answer': ans}

class EvalDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = load_image(row['image_path'], CONFIG['img_size'])
        ans = int(row['answer']) if pd.notna(row.get('answer')) else 0
        ans = min(ans, 4)
        return {'image': img, 'text': build_prompt(row, include_answer=False), 'answer': ans}

# ── Collate ───────────────────────────────────────────────────────────────────
def build_collate_fn(proc):
    LETTERS        = ['A', 'B', 'C', 'D', 'E']
    answer_tok_ids = proc.tokenizer.convert_tokens_to_ids(LETTERS)
    assert all(isinstance(t, int) for t in answer_tok_ids),         "A/B/C/D/E must each be a single token!"

    def collate_fn(batch):
        inputs = proc(
            text=[b['text'] for b in batch],
            images=[b['image'] for b in batch],
            return_tensors='pt', padding=True,
            max_length=CONFIG['max_length'], truncation=True,
        )
        labels = torch.full_like(inputs['input_ids'], fill_value=-100)
        for i, b in enumerate(batch):
            ans_idx   = min(int(b['answer']), len(answer_tok_ids) - 1)
            target_id = answer_tok_ids[ans_idx]
            seq       = inputs['input_ids'][i]
            real_end  = int((seq != proc.tokenizer.pad_token_id).sum()) - 1
            for pos in range(real_end, max(-1, real_end - 10), -1):
                if seq[pos].item() == target_id:
                    labels[i, pos] = target_id
                    break
            else:
                labels[i, real_end] = target_id
        inputs['labels'] = labels
        return inputs, torch.tensor([b['answer'] for b in batch], dtype=torch.long)

    return collate_fn

# ── Model ─────────────────────────────────────────────────────────────────────
base_model = AutoModelForVision2Seq.from_pretrained(
    CONFIG['model_id'],
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True),
    device_map='auto', low_cpu_mem_usage=True)
base_model = prepare_model_for_kbit_training(base_model)
base_model.config.use_cache = False

if CONFIG['resume_from']:
    model = PeftModel.from_pretrained(
        base_model, Path(CONFIG['resume_from']), is_trainable=True)
    print(f"Resumed: {CONFIG['resume_from']}")
else:
    model = get_peft_model(base_model, LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=CONFIG['lora_r'], lora_alpha=CONFIG['lora_alpha'],
        lora_dropout=CONFIG['lora_dropout'],
        target_modules=CONFIG['lora_targets'], bias='none'))
model.print_trainable_parameters()

# ── DataLoaders ───────────────────────────────────────────────────────────────
collate_fn   = build_collate_fn(processor)
train_loader = DataLoader(
    ScienceQADataset(train_df),
    batch_size=CONFIG['batch_size'], shuffle=True,
    collate_fn=collate_fn, num_workers=0, pin_memory=False, drop_last=True)

# ── Optimizer + scheduler ─────────────────────────────────────────────────────
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer  = AdamW(trainable_params, lr=CONFIG['learning_rate'],
                   weight_decay=CONFIG['weight_decay'])
total_steps  = (len(train_loader) // CONFIG['grad_accum']) * CONFIG['num_epochs']
warmup_steps = max(50, int(CONFIG['warmup_ratio'] * total_steps))

def lr_lambda(s):
    if s < warmup_steps: return s / max(1, warmup_steps)
    p = (s - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))

scheduler = LambdaLR(optimizer, lr_lambda)
if CONFIG['resume_from'] and START_EPOCH > 0:
    steps_done = (len(train_loader) // CONFIG['grad_accum']) * START_EPOCH
    for _ in range(steps_done): scheduler.step()
    print(f"LR scheduler fast-forwarded {steps_done} steps")

print(f"Steps/epoch: {len(train_loader)} · Total: {total_steps} · Warmup: {warmup_steps}")

# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(df, n=None, desc='Eval'):
    model.eval()
    if n is not None: df = df.sample(n=min(n, len(df)), random_state=CONFIG['seed'])
    ds = EvalDataset(df)
    def eval_collate(batch):
        inputs = processor(
            text=[b['text'] for b in batch],
            images=[b['image'] for b in batch],
            return_tensors='pt', padding=True,
            max_length=CONFIG['max_length'], truncation=True)
        return inputs, torch.tensor([b['answer'] for b in batch], dtype=torch.long)
    loader = DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=False,
                        collate_fn=eval_collate, num_workers=0, pin_memory=False)
    ans_tok = torch.tensor(
        processor.tokenizer.convert_tokens_to_ids(['A','B','C','D','E']),
        device=DEVICE)
    correct, total = 0, 0
    for inputs, answers in tqdm(loader, desc=desc, leave=False):
        ei = {k: v.to(DEVICE) for k, v in inputs.items()}
        answers = answers.to(DEVICE)
        with autocast('cuda', dtype=COMPUTE_DTYPE):
            logits = model(**ei).logits
        last_pos    = ei['attention_mask'].sum(dim=1) - 1
        last_logits = logits[torch.arange(logits.size(0), device=DEVICE), last_pos]
        preds       = last_logits[:, ans_tok].argmax(dim=-1)
        correct += (preds == answers).sum().item()
        total   += answers.size(0)
    model.train()
    return correct / total if total > 0 else 0.0

# ── Chart ─────────────────────────────────────────────────────────────────────
CHART_STYLE = {
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#3a3d4f', 'axes.labelcolor': '#c8ccd8',
    'xtick.color': '#7a7e93', 'ytick.color': '#7a7e93',
    'grid.color': '#2a2d3f', 'grid.linestyle': '--', 'grid.alpha': 0.6,
    'text.color': '#c8ccd8', 'font.family': 'monospace',
}

def save_epoch_chart(step_losses, epoch_accs, epoch_lrs, run_id, out_dir):
    charts_dir = out_dir / 'charts'
    charts_dir.mkdir(exist_ok=True)
    with plt.rc_context(CHART_STYLE):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(f'Run: {run_id}', fontsize=11, color='#e0e3f0', y=1.02)
        ax = axes[0]
        steps  = [s for s, _ in step_losses]
        losses = [l for _, l in step_losses]
        ema, alpha, v = [], 0.05, (losses[0] if losses else 0)
        for l in losses:
            v = alpha * l + (1 - alpha) * v; ema.append(v)
        ax.plot(steps, losses, color='#2d3561', linewidth=0.6, alpha=0.5, label='raw')
        ax.plot(steps, ema,    color='#6c8ef5', linewidth=1.8, label='EMA')
        for ep_n in range(1, CONFIG['num_epochs'] + 1):
            ax.axvline(ep_n * len(train_loader), color='#ff6b6b',
                       linewidth=0.8, linestyle=':', alpha=0.7)
        ax.set_title('Training Loss', fontsize=9); ax.set_xlabel('Step', fontsize=8)
        ax.set_ylabel('Loss', fontsize=8); ax.legend(fontsize=7); ax.grid(True)
        ax = axes[1]
        if epoch_accs:
            ep_nums = [e for e, _ in epoch_accs]; accs = [a for _, a in epoch_accs]
            ax.plot(ep_nums, accs, color='#50e3c2', linewidth=2.0, marker='o', markersize=6)
            ax.axhline(0.33, color='#ff6b6b', linewidth=1.0, linestyle='--', label='random')
            for e, a in zip(ep_nums, accs):
                ax.annotate(f'{a:.1%}', (e, a), textcoords='offset points',
                            xytext=(0, 8), ha='center', fontsize=8, color='#50e3c2')
            ax.set_ylim(0.0, 1.0)
            ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
        ax.set_title('Quick-Val Accuracy', fontsize=9); ax.set_xlabel('Epoch', fontsize=8)
        ax.set_ylabel('Accuracy', fontsize=8); ax.legend(fontsize=7); ax.grid(True)
        ax = axes[2]
        if epoch_lrs:
            ep_nums = [e for e, _ in epoch_lrs]; lrvs = [l for _, l in epoch_lrs]
            ax.plot(ep_nums, lrvs, color='#f5a623', linewidth=2.0, marker='s', markersize=5)
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1e'))
        ax.set_title('Learning Rate', fontsize=9); ax.set_xlabel('Epoch', fontsize=8)
        ax.set_ylabel('LR', fontsize=8); ax.grid(True)
        plt.tight_layout()
        chart_path = charts_dir / f'training_e{len(epoch_accs):02d}.png'
        plt.savefig(chart_path, dpi=130, bbox_inches='tight', facecolor=fig.get_facecolor())
        plt.show(); plt.close(fig)
        print(f"  \U0001f4ca Chart \u2192 {chart_path.name}")

# ── Training loop ─────────────────────────────────────────────────────────────
run_start   = time.time()
epoch_log   = {}
step_losses = []
epoch_accs  = []
epoch_lrs   = []
checkpoints = []
model.train()
global_step = (len(train_loader) // CONFIG['grad_accum']) * START_EPOCH

for epoch in range(START_EPOCH + 1, CONFIG['num_epochs'] + 1):
    epoch_loss = 0.0
    t0 = time.time()
    optimizer.zero_grad()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f"Epoch {epoch}/{CONFIG['num_epochs']}")
    for step, (inputs, _answers) in pbar:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with autocast('cuda', dtype=COMPUTE_DTYPE):
            loss = model(**inputs).loss / CONFIG['grad_accum']
        if torch.isnan(loss):
            print(f"  [step {step}] NaN loss — skipping"); optimizer.zero_grad(); continue
        if USE_SCALER: scaler.scale(loss).backward()
        else:          loss.backward()
        epoch_loss += loss.item() * CONFIG['grad_accum']
        if (step + 1) % CONFIG['grad_accum'] == 0:
            if USE_SCALER: scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, CONFIG['max_grad_norm'])
            if USE_SCALER: scaler.step(optimizer); scaler.update()
            else:          optimizer.step()
            scheduler.step(); optimizer.zero_grad(); global_step += 1
            step_losses.append((global_step, epoch_loss / max(1, step + 1)))
        if step % 50 == 0:
            pbar.set_postfix(loss=f"{epoch_loss/max(1,step+1):.4f}",
                             lr=f"{scheduler.get_last_lr()[0]:.2e}")
    elapsed  = (time.time() - t0) / 60
    avg_loss = epoch_loss / len(train_loader)
    cur_lr   = scheduler.get_last_lr()[0]
    qv = evaluate(val_df, n=CONFIG['quick_val_n'], desc=f'Quick-val e{epoch}')
    flag = '  \u26a0 <30%!' if qv < 0.30 else ''
    print(f"Epoch {epoch}/{CONFIG['num_epochs']} | loss {avg_loss:.4f} | "
          f"quick_val {qv:.1%}{flag} | {elapsed:.1f} min")
    epoch_accs.append((epoch, qv))
    epoch_lrs.append((epoch, cur_lr))
    epoch_log[f'epoch_{epoch}'] = {
        'loss': round(avg_loss, 4), 'quick_val': round(qv, 4),
        'lr': cur_lr, 'time_min': round(elapsed, 1),
    }
    with open(LOG_FILE, 'a') as f:
        f.write(json.dumps({'run_id': CONFIG['run_id'], 'epoch': epoch,
                             **epoch_log[f'epoch_{epoch}']}) + '\n')
    ckpt_dir = OUT_DIR / f"checkpoint_epoch{epoch}"
    model.save_pretrained(ckpt_dir); processor.save_pretrained(ckpt_dir)
    checkpoints.append(ckpt_dir)
    print(f"  \u2713 Checkpoint \u2192 {ckpt_dir.name}")
    save_epoch_chart(step_losses, epoch_accs, epoch_lrs, CONFIG['run_id'], OUT_DIR)
    if qv < 0.30:
        print("  \u2192 Aborting. Fix config before retrying."); break

total_min = (time.time() - run_start) / 60
print(f"\nTotal training time: {total_min:.1f} min")
print("Run Cell 4 \u2192 full val + save final adapter + cleanup.")


Mixed precision: BF16 (A100)
Device: cuda · NVIDIA A100-SXM4-40GB
Train: 3,109 · Val: 1,048
Captions loaded: 3026/7266 non-empty
trainable params: 9,568,256 || all params: 517,050,560 || trainable%: 1.8505
Steps/epoch: 388 · Total: 194 · Warmup: 50


Epoch 1/2:   0%|          | 0/388 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Quick-val e1:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 1/2 | loss 1.1254 | quick_val 62.7% | 41.1 min
  ✓ Checkpoint → checkpoint_epoch1
  📊 Chart → training_e01.png


Epoch 2/2:   0%|          | 0/388 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Quick-val e2:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 2/2 | loss 0.5295 | quick_val 65.3% | 41.1 min
  ✓ Checkpoint → checkpoint_epoch2
  📊 Chart → training_e02.png

Total training time: 85.0 min
Run Cell 4 → full val + save final adapter + cleanup.


In [ ]:
# ── CELL 4 · FULL VAL + SAVE ──────────────────────────────────────────────────
import shutil

last_epoch_key = f"epoch_{CONFIG['num_epochs']}"
last_qv        = epoch_log.get(last_epoch_key, {}).get('quick_val', 0.0)

if last_qv < 0.35:
    print(f"quick_val {last_qv:.1%} < 35% — skipping full val. Fix config.")
    full_val_acc = None
else:
    print(f"quick_val {last_qv:.1%} ≥ 35% → running full val on {len(val_df):,} examples …")
    full_val_acc = evaluate(val_df, desc='Full-val')
    print(f"Full-val accuracy: {full_val_acc:.4f} ({full_val_acc:.1%})")

# Save final adapter
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print(f"\n✓ Adapter saved → {ADAPTER_DIR}")

# Delete per-epoch checkpoints
for ckpt in checkpoints:
    if ckpt.exists():
        shutil.rmtree(ckpt)
        print(f"  Deleted checkpoint → {ckpt.name}")

# ── Rich results JSON (includes all per-step + per-epoch data for aggregation) ──
run_results = {
    'run_id':         CONFIG['run_id'],
        'config': {k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()},
    'epoch_log':      epoch_log,
    'full_val_acc':   round(full_val_acc, 4) if full_val_acc is not None else None,
    'total_time_min': round(total_min, 1),
    'adapter_dir':    str(ADAPTER_DIR),
    'status':         'done',
    # ── Chart data (loaded by Cell 6 aggregator to rebuild any chart) ──────────
    'chart_data': {
        'step_losses':  step_losses,    # [[global_step, loss], ...]
        'epoch_accs':   epoch_accs,     # [[epoch, acc], ...]
        'epoch_lrs':    epoch_lrs,      # [[epoch, lr], ...]
    },
}

results_path = OUT_DIR / f"results_{CONFIG['run_id']}.json"
results_path.write_text(json.dumps(run_results, indent=2))
print(f"✓ Results  → {results_path}")
print()
print(json.dumps({
    'run_id':       run_results['run_id'],
    'epoch_log':    run_results['epoch_log'],
    'full_val_acc': run_results['full_val_acc'],
}, indent=2))


quick_val 65.3% ≥ 35% → running full val on 1,048 examples …


Full-val:   0%|          | 0/131 [00:00<?, ?it/s]

Full-val accuracy: 0.7032 (70.3%)

✓ Adapter saved → /content/drive/MyDrive/scienceqa_runs/run_09/adapter_run_09
  Deleted checkpoint → checkpoint_epoch1
  Deleted checkpoint → checkpoint_epoch2
✓ Results  → /content/drive/MyDrive/scienceqa_runs/run_09/results_run_09.json

{
  "run_id": "run_09",
  "epoch_log": {
    "epoch_1": {
      "loss": 1.1254,
      "quick_val": 0.6267,
      "lr": 0.00015187732581605217,
      "time_min": 41.1
    },
    "epoch_2": {
      "loss": 0.5295,
      "quick_val": 0.6533,
      "lr": 0.0,
      "time_min": 41.1
    }
  },
  "full_val_acc": 0.7032
}


In [3]:
# ── CELL 5 · INFERENCE ────────────────────────────────────────────────────────
# Fully self-contained — safe to run after a runtime reload on T4.
# Workflow: finish training → change runtime to T4 → Cell 1 → Cell 2 → Cell 5

import json, gc
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.cuda.amp import autocast
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import PeftModel

# ── Device + dtype ────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
COMPUTE_DTYPE = (
    torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
print(f"Device: {DEVICE} · dtype: {COMPUTE_DTYPE}")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = Path(CONFIG['data_dir'])
OUT_DIR     = Path(CONFIG['output_dir']) / CONFIG['run_id']
ADAPTER_DIR = OUT_DIR / f"adapter_{CONFIG['run_id']}"

CAPTION_CSV = DATA_DIR / 'captions.csv'

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(
        f"Adapter not found: {ADAPTER_DIR}\n"
        f"Make sure Cell 4 completed for run_id='{CONFIG['run_id']}'"
    )
print(f"Loading adapter: {ADAPTER_DIR}")

# ── Load processor + model ────────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(ADAPTER_DIR)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

base_model = AutoModelForVision2Seq.from_pretrained(
    CONFIG['model_id'],
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True),
    device_map='auto', low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("Model ready.")

# ── Helpers (redefined — not in memory after reload) ──────────────────────────
MAX_LECTURE_CHARS = 800

def load_df(name):
    df = pd.read_csv(DATA_DIR / f'{name}.csv')
    df['choices'] = df['choices'].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x)
    return df

# Load captions
if CAPTION_CSV.exists():
    captions_df = pd.read_csv(CAPTION_CSV).set_index('id')
    print(f"Captions: {len(captions_df)} rows")
else:
    captions_df = None
    print("\u26a0  No captions.csv — running without caption augmentation")

def load_image(image_path, size):
    try:
        img = Image.open(DATA_DIR / image_path).convert('RGB')
        img.thumbnail((size, size), Image.LANCZOS)
        padded = Image.new('RGB', (size, size), (128, 128, 128))
        padded.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
        return padded
    except Exception:
        return Image.new('RGB', (size, size), (128, 128, 128))

def build_prompt(row, include_answer=False):
    parts = []
    grade   = row.get('grade', '')
    subject = row.get('subject', '')
    if grade and str(grade) not in ('', 'nan'):
        parts.append(f"Grade {grade} \u00b7 {subject}")
    if captions_df is not None:
        try:
            cap = captions_df.loc[row['id'], 'caption']
            if cap and str(cap) not in ('', 'nan'):
                parts.append(f"Caption: {str(cap).strip()}")
        except KeyError:
            pass
    hint    = str(row.get('hint',    '')).strip()
    lecture = str(row.get('lecture', '')).strip()
    hint    = '' if hint    in ('', 'nan') else hint
    lecture = '' if lecture in ('', 'nan') else lecture[:MAX_LECTURE_CHARS]
    ctx = [c for c in [hint, lecture] if c]
    if ctx:
        parts.append('Context:\n' + '\n'.join(ctx))
    parts.append(f"Question: {row['question']}")
    parts.append('Choices:\n' + '\n'.join(
        f'  {chr(65+i)}. {c}' for i, c in enumerate(row['choices'])))
    parts.append('Answer with the option\u2019s letter from the given choices directly.')
    parts.append('Answer:')
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": '\n'.join(parts)},
    ]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True)

class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {'image': load_image(row['image_path'], CONFIG['img_size']),
                'text':  build_prompt(row), 'answer': 0}

def infer_collate(batch):
    inputs = processor(
        text=[b['text'] for b in batch],
        images=[b['image'] for b in batch],
        return_tensors='pt', padding=True,
        max_length=CONFIG['max_length'], truncation=True)
    return inputs, torch.zeros(len(batch), dtype=torch.long)

# ── Run inference ─────────────────────────────────────────────────────────────
test_df = load_df('test')
print(f"Test: {len(test_df):,} examples")

loader = DataLoader(TestDataset(test_df), batch_size=CONFIG['batch_size'],
                    shuffle=False, collate_fn=infer_collate,
                    num_workers=0, pin_memory=False)

ans_tok_ids = processor.tokenizer.convert_tokens_to_ids(['A','B','C','D','E'])
ans_tok_tensor = torch.tensor(ans_tok_ids, device=DEVICE)  # (5,)

preds = []
for i, (inputs, _) in enumerate(tqdm(loader, desc='Inference')):
    inps = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad(), autocast(dtype=COMPUTE_DTYPE):
        logits = model(**inps).logits                             # (B, S, V)
    last_pos    = inps['attention_mask'].sum(dim=1) - 1          # (B,)
    last_logits = logits[torch.arange(logits.size(0), device=DEVICE), last_pos]  # (B, V)
    ans_logits  = last_logits[:, ans_tok_tensor]                  # (B, 5)
    batch_preds = ans_logits.argmax(dim=-1).cpu().tolist()

    base = i * CONFIG['batch_size']
    for j, pred in enumerate(batch_preds):
        idx = base + j
        if idx < len(test_df):
            preds.append({'id': test_df.iloc[idx]['id'], 'answer': int(pred)})

sub_df   = pd.DataFrame(preds)
sub_path = OUT_DIR / f"submission_{CONFIG['run_id']}.csv"
sub_df.to_csv(sub_path, index=False)
print(f"\n✓ Submission ({len(sub_df)} rows) → {sub_path}")
print(sub_df['answer'].value_counts().sort_index().rename(
    {0:'A',1:'B',2:'C',3:'D',4:'E'}))

del model, base_model
gc.collect(); torch.cuda.empty_cache()
print("Done.")


Device: cuda · dtype: torch.bfloat16
Loading adapter: /content/drive/MyDrive/scienceqa_runs/run_09/adapter_run_09


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model ready.
Captions: 7266 rows
Test: 1,008 examples


Inference:   0%|          | 0/126 [00:00<?, ?it/s]

/tmp/ipykernel_12136/2088643551.py:145: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(dtype=COMPUTE_DTYPE):



✓ Submission (1008 rows) → /content/drive/MyDrive/scienceqa_runs/run_09/submission_run_09.csv
answer
A    340
B    238
C    329
D     78
E     23
Name: count, dtype: int64
Done.


In [ ]:
# ── CELL 6 · AGGREGATE CHART — compare all runs ───────────────────────────────
# Scans output_dir for every results_*.json and draws a unified comparison.
# Run this any time — it reads what's already saved on Drive.
#
# Saves to:  output_dir/aggregate_chart.png
#            output_dir/aggregate_summary.json

import json, math
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.cm as cm

OUTPUT_DIR = Path(CONFIG['output_dir'])

# ── Collect all results files ─────────────────────────────────────────────────
result_files = sorted(OUTPUT_DIR.rglob('results_*.json'))
print(f"Found {len(result_files)} result file(s):")
for f in result_files:
    print(f"  {f.relative_to(OUTPUT_DIR)}")

if not result_files:
    print("No results yet — run at least one training session first.")
else:
    runs = []
    for fp in result_files:
        try:
            data = json.loads(fp.read_text())
            runs.append(data)
        except Exception as e:
            print(f"  ⚠ Could not load {fp.name}: {e}")

    # ── Sort by full_val_acc descending ───────────────────────────────────────
    runs.sort(key=lambda r: r.get('full_val_acc') or 0, reverse=True)

    CHART_STYLE = {
        'figure.facecolor': '#0f1117',
        'axes.facecolor':   '#1a1d27',
        'axes.edgecolor':   '#3a3d4f',
        'axes.labelcolor':  '#c8ccd8',
        'xtick.color':      '#7a7e93',
        'ytick.color':      '#7a7e93',
        'grid.color':       '#2a2d3f',
        'grid.linestyle':   '--',
        'grid.alpha':       0.6,
        'text.color':       '#c8ccd8',
        'font.family':      'monospace',
    }

    # Colour palette — one colour per run, cycles if > 10
    palette = [
        '#6c8ef5','#50e3c2','#f5a623','#ff6b6b',
        '#b8e986','#bd10e0','#9b59b6','#1abc9c',
        '#e67e22','#e74c3c',
    ]

    with plt.rc_context(CHART_STYLE):
        fig = plt.figure(figsize=(18, 12))
        fig.suptitle('ScienceQA — All Runs Comparison', fontsize=13,
                     color='#e0e3f0', y=0.98)

        # Grid: 2 rows × 3 cols
        # [0,0] Loss curves   [0,1] Quick-val per epoch  [0,2] Final val bar
        # [1,0] LR curves     [1,1] Loss vs val scatter   [1,2] Summary table
        gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)
        ax_loss   = fig.add_subplot(gs[0, 0])
        ax_acc    = fig.add_subplot(gs[0, 1])
        ax_bar    = fig.add_subplot(gs[0, 2])
        ax_lr     = fig.add_subplot(gs[1, 0])
        ax_scat   = fig.add_subplot(gs[1, 1])
        ax_table  = fig.add_subplot(gs[1, 2])

        for ax in [ax_loss, ax_acc, ax_bar, ax_lr, ax_scat]:
            ax.grid(True)

        # ── (0,0) Training loss curves ────────────────────────────────────────
        for i, run in enumerate(runs):
            cd    = run.get('chart_data', {})
            sls   = cd.get('step_losses', [])
            color = palette[i % len(palette)]
            label = run['run_id']
            if sls:
                steps  = [s for s, _ in sls]
                losses = [l for _, l in sls]
                # EMA smooth
                ema, alpha, v = [], 0.05, losses[0]
                for l in losses:
                    v = alpha * l + (1 - alpha) * v
                    ema.append(v)
                ax_loss.plot(steps, ema, color=color, linewidth=1.6, label=label)

        ax_loss.set_title('Training Loss (EMA)', fontsize=9)
        ax_loss.set_xlabel('Step', fontsize=8)
        ax_loss.set_ylabel('Loss', fontsize=8)
        ax_loss.legend(fontsize=7, loc='upper right')

        # ── (0,1) Quick-val accuracy per epoch ───────────────────────────────
        ax_acc.axhline(0.33, color='#ff6b6b', linewidth=1.0, linestyle='--',
                       label='random baseline')
        for i, run in enumerate(runs):
            cd    = run.get('chart_data', {})
            eas   = cd.get('epoch_accs', [])
            color = palette[i % len(palette)]
            if eas:
                eps  = [e for e, _ in eas]
                accs = [a for _, a in eas]
                ax_acc.plot(eps, accs, color=color, linewidth=1.8,
                            marker='o', markersize=5, label=run['run_id'])
                ax_acc.annotate(f"{accs[-1]:.1%}", (eps[-1], accs[-1]),
                                textcoords='offset points', xytext=(4, 0),
                                fontsize=7, color=color)

        ax_acc.set_title('Quick-Val Accuracy / Epoch', fontsize=9)
        ax_acc.set_xlabel('Epoch', fontsize=8)
        ax_acc.set_ylabel('Accuracy', fontsize=8)
        ax_acc.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
        ax_acc.set_ylim(0.0, 1.0)
        ax_acc.legend(fontsize=7, loc='lower right')

        # ── (0,2) Final full-val bar chart ────────────────────────────────────
        bar_runs = [r for r in runs if r.get('full_val_acc') is not None]
        if bar_runs:
            names = [r['run_id'] for r in bar_runs]
            vals  = [r['full_val_acc'] for r in bar_runs]
            bars  = ax_bar.barh(names, vals,
                                color=[palette[i % len(palette)] for i in range(len(bar_runs))],
                                height=0.5, edgecolor='none')
            ax_bar.axvline(0.33, color='#ff6b6b', linewidth=1.0, linestyle='--',
                           label='random')
            for bar, val in zip(bars, vals):
                ax_bar.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                            f'{val:.1%}', va='center', fontsize=8,
                            color='#c8ccd8')
            ax_bar.set_xlim(0, 1.0)
            ax_bar.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
            ax_bar.set_title('Full-Val Accuracy', fontsize=9)
            ax_bar.set_xlabel('Accuracy', fontsize=8)
            ax_bar.legend(fontsize=7)
        else:
            ax_bar.text(0.5, 0.5, 'No full-val\nresults yet',
                        ha='center', va='center', fontsize=9,
                        transform=ax_bar.transAxes, color='#7a7e93')
            ax_bar.set_title('Full-Val Accuracy', fontsize=9)

        # ── (1,0) LR curves ───────────────────────────────────────────────────
        for i, run in enumerate(runs):
            cd    = run.get('chart_data', {})
            lrs   = cd.get('epoch_lrs', [])
            color = palette[i % len(palette)]
            if lrs:
                eps  = [e for e, _ in lrs]
                lrvs = [l for _, l in lrs]
                ax_lr.plot(eps, lrvs, color=color, linewidth=1.8,
                           marker='s', markersize=4, label=run['run_id'])

        ax_lr.set_title('Learning Rate / Epoch', fontsize=9)
        ax_lr.set_xlabel('Epoch', fontsize=8)
        ax_lr.set_ylabel('LR', fontsize=8)
        ax_lr.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1e'))
        ax_lr.legend(fontsize=7, loc='upper right')

        # ── (1,1) Final loss vs val-acc scatter ───────────────────────────────
        for i, run in enumerate(runs):
            el     = run.get('epoch_log', {})
            color  = palette[i % len(palette)]
            epochs = sorted(el.keys())
            if epochs:
                last_ep = el[epochs[-1]]
                loss_v  = last_ep.get('loss')
                acc_v   = last_ep.get('quick_val')
                if loss_v is not None and acc_v is not None:
                    ax_scat.scatter(loss_v, acc_v, color=color, s=80, zorder=5)
                    ax_scat.annotate(run['run_id'], (loss_v, acc_v),
                                     textcoords='offset points', xytext=(5, 3),
                                     fontsize=7, color=color)

        ax_scat.axhline(0.33, color='#ff6b6b', linewidth=1.0, linestyle='--')
        ax_scat.set_title('Final Loss vs Quick-Val Acc', fontsize=9)
        ax_scat.set_xlabel('Train Loss (last epoch)', fontsize=8)
        ax_scat.set_ylabel('Quick-Val Acc', fontsize=8)
        ax_scat.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

        # ── (1,2) Summary table ───────────────────────────────────────────────
        ax_table.axis('off')
        headers = ['Run', 'r', 'lr', 'ep1 qv', 'ep2 qv', 'full val']
        rows    = []
        for run in runs:
            cfg = run.get('config', {})
            el  = run.get('epoch_log', {})
            rows.append([
                run['run_id'],
                str(cfg.get('lora_r', '?')),
                str(cfg.get('learning_rate', '?')),
                f"{el.get('epoch_1', {}).get('quick_val', float('nan')):.1%}"
                    if 'epoch_1' in el else '—',
                f"{el.get('epoch_2', {}).get('quick_val', float('nan')):.1%}"
                    if 'epoch_2' in el else '—',
                f"{run['full_val_acc']:.1%}"
                    if run.get('full_val_acc') is not None else '—',
            ])

        tbl = ax_table.table(
            cellText=rows, colLabels=headers,
            cellLoc='center', loc='center',
            bbox=[0.0, 0.05, 1.0, 0.90],
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8)
        # Style header
        for j in range(len(headers)):
            tbl[0, j].set_facecolor('#2d3561')
            tbl[0, j].set_text_props(color='#e0e3f0', fontweight='bold')
        # Alternate row colours
        for r_i in range(1, len(rows) + 1):
            bg = '#1a1d27' if r_i % 2 == 0 else '#20243a'
            for c_i in range(len(headers)):
                tbl[r_i, c_i].set_facecolor(bg)
                tbl[r_i, c_i].set_text_props(color='#c8ccd8')
        ax_table.set_title('Summary', fontsize=9, pad=8)

        # ── Save ──────────────────────────────────────────────────────────────
        agg_chart = OUTPUT_DIR / 'aggregate_chart.png'
        plt.savefig(agg_chart, dpi=150, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        plt.show()
        plt.close(fig)
        print(f"\n✓ Aggregate chart → {agg_chart}")

    # ── Save aggregate summary JSON ───────────────────────────────────────────
    summary = [{
        'run_id':     r['run_id'],
        'full_val_acc': r.get('full_val_acc'),
        'epoch_log':    r.get('epoch_log', {}),
        'config_key':   {k: r.get('config', {}).get(k)
                         for k in ['lora_r','lora_alpha','learning_rate',
                                   'lora_targets','img_size','num_epochs']},
    } for r in runs]

    agg_json = OUTPUT_DIR / 'aggregate_summary.json'
    agg_json.write_text(json.dumps(summary, indent=2))
    print(f"✓ Aggregate summary → {agg_json}")
    print()
    for r in runs:
        fv = r.get('full_val_acc')
        print(f"  {r['run_id']:20s}  full_val={f'{fv:.1%}' if fv else '—':>7s}  "
              f"epochs_done={len(r.get('epoch_log', {}))}")
